In [1]:
from rdkit import Chem
from pathlib import Path

In [2]:
mol2_dir1 = Path("D:\\DESKTOP\\Current-Projects\\tbrucei_docking_data\\positive_tbrucie_mol2")
mol2_dir2 = Path("D:\\DESKTOP\\Current-Projects\\tbrucei_docking_data\\negative_mamalian_mol2")

sources = [
    (mol2_dir1, 'positive'),
    (mol2_dir2, 'negative'),
]

molecules = {}
failed_files = []
for mol2_dir, label in sources:
    for mol2_file in mol2_dir.glob("*.mol2"):
        try:
            mol = Chem.MolFromMol2File(
                str(mol2_file),
                sanitize=True,
                removeHs=False
                )
            if mol is None:
                failed_files.append(mol2_file.name)
                continue

            molecules[mol2_file.stem] = mol
        except Exception as e:
            failed_files.append((mol2_file.name, str(e)))

print("Successful: ", len(molecules))
print("Failed: ", len(failed_files))

[20:06:17] ZINC000001702786: warning - aromatic N with 3 aromatic bonds - skipping charge guess for this atom
[20:06:17] Can't kekulize mol.  Unkekulized atoms: 1 2 3 4 5 6 7 8 9 10 11 12 13 14 16 17 18
[20:06:17] sanitize [20:06:17] ZINC000001702786: [20:06:18] ZINC000002271427: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.
[20:06:19] ZINC000004290180: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.
[20:06:20] ZINC000005543098: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.
[20:06:21] ZINC000009041729: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.
[20:06:26] ZINC000082204980: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.
[20:06:26] ZINC000088267072: Warning - no explicit hydrogens in mol2 file but needed for formal charge estimation.
[20:06:26] ZINC000095681876: Warning - no explicit hydroge

Successful:  12966
Failed:  1


In [3]:
print(failed_files)

['ZINC000001702786.mol2']


In [13]:
# Generate Bemis-Murcko scaffolds
from rdkit.Chem.Scaffolds import MurckoScaffold

scaffold_dict = {}

for molecule_id, mol in molecules.items():
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    scaffold_smiles = Chem.MolToSmiles(
        scaffold,
        canonical=True
    )
    scaffold_dict[molecule_id] = scaffold_smiles

#print(list(scaffold_dict.items())[:10])

In [14]:
scaffold_dict

{'ZINC000000000910': 'c1ccc(-c2ccsn2)cc1',
 'ZINC000000001690': 'c1ccc2c(c1)Sc1ccccc1S2',
 'ZINC000000029877': 'c1ccccc1',
 'ZINC000000033416': 'O=C(Nc1ccccc1)c1ccn[nH]1',
 'ZINC000000034999': 'O=C1CC(c2ccccc2)OC(=O)C12CCCCC2',
 'ZINC000000040506': 'O=C1c2nccnc2C(=O)N1c1ccccc1',
 'ZINC000000041501': 'O=C(Nc1ccccc1)N1CCSC1',
 'ZINC000000041615': 'O=C1c2ccccc2C(=O)N1c1ccccc1',
 'ZINC000000041676': 'O=C1c2ccccc2-c2nc3ccccc3n21',
 'ZINC000000045096': 'O=C1CCCC2=Nc3ccccc3N=C(c3ccccc3)C12',
 'ZINC000000045614': 'O=C(Nc1ccccc1)c1ccoc1',
 'ZINC000000051702': 'c1ccc(-n2ccnn2)cc1',
 'ZINC000000052123': 'O=C1C2C3C=CC(C3=C3CCCC3)C2C(=O)N1Cc1ccccc1',
 'ZINC000000056297': 'c1ccc(-c2cn3ccccc3n2)cc1',
 'ZINC000000056550': 'c1ccccc1',
 'ZINC000000056839': 'O=c1[n-]cnc2[nH]cnc12',
 'ZINC000000057304': 'c1ccc(C2Nc3ccccc3-c3ccccc32)cc1',
 'ZINC000000062898': 'O=C1c2cccc3cccc(c23)C(=O)N1c1ccccc1',
 'ZINC000000063319': 'O=C(/C=C/c1ccc(-c2ccccc2)o1)N1CCCCC1',
 'ZINC000000063520': 'O=C1c2ccc3c4c(ccc(c24)C(=O)

In [17]:
import pandas as pd
df_scaffold = pd.DataFrame(scaffold_dict.items(), columns=['Molecule', 'Scaffold'])
print('Shape of Scaffold :', df_scaffold.shape)

Shape of Scaffold : (12966, 2)


In [20]:
df_scaffold.head(10)

,Molecule,Scaffold
0,ZINC000000000910,c1ccc(-c2ccsn2)cc1
1,ZINC000000001690,c1ccc2c(c1)Sc1ccccc1S2
2,ZINC000000029877,c1ccccc1
3,ZINC000000033416,O=C(Nc1ccccc1)c1ccn[nH]1
4,ZINC000000034999,O=C1CC(c2ccccc2)OC(=O)C12CCCCC2
5,ZINC000000040506,O=C1c2nccnc2C(=O)N1c1ccccc1
6,ZINC000000041501,O=C(Nc1ccccc1)N1CCSC1
7,ZINC000000041615,O=C1c2ccccc2C(=O)N1c1ccccc1
8,ZINC000000041676,O=C1c2ccccc2-c2nc3ccccc3n21
9,ZINC000000045096,O=C1CCCC2=Nc3ccccc3N=C(c3ccccc3)C12


In [21]:
# Save to a file
df_scaffold.to_csv('scaffold_rdkit.csv', index=False)